In [1]:
import os
from appwrite.client import Client
from appwrite.services.storage import Storage
from appwrite.services.databases import Databases
from appwrite.input_file import InputFile
import pandas as pd
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

True

Environment Variables:
Endpoint: https://fra.cloud.appwrite.io/v1
Project ID: 6845ae7f000700b9ce1e
Bucket ID: 6845af22003dfe2a38b9
Database ID: 6845af5d00016a672555
Collection ID: 6845af7000180f111df0


NameError: name 'APPWRITE_ENDPOINT' is not defined

In [4]:
# Initialize Appwrite Client
client = Client()
client.set_endpoint(os.getenv('APPWRITE_ENDPOINT'))
client.set_project(os.getenv('APPWRITE_PROJECT_ID'))
client.set_key(os.getenv('APPWRITE_API_KEY'))

In [5]:
storage = Storage(client)
database = Databases(client)

In [6]:
# Constants
BUCKET_ID = os.getenv('APPWRITE_BUCKET_ID')
DATABASE_ID = os.getenv('APPWRITE_DATABASE_ID')
COLLECTION_ID = os.getenv('APPWRITE_COLLECTION_ID')


In [7]:

# Read product data
df = pd.read_json('products/products.jsonl', lines=True)


Error uploading product Cappuccino: Invalid document structure: Unknown attribute: "name"
Error uploading product Jumbo Savory Scone: Invalid document structure: Unknown attribute: "name"
Error uploading product Latte: Invalid document structure: Unknown attribute: "name"
Error uploading product Chocolate Chip Biscotti: Invalid document structure: Unknown attribute: "name"
Error uploading product Espresso shot: Invalid document structure: Unknown attribute: "name"
Error uploading product Hazelnut Biscotti: Invalid document structure: Unknown attribute: "name"
Error uploading product Chocolate Croissant: Invalid document structure: Unknown attribute: "name"
Error uploading product Dark chocolate: Invalid document structure: Unknown attribute: "name"
Error uploading product Cranberry Scone: Invalid document structure: Unknown attribute: "name"
Error uploading product Croissant: Invalid document structure: Unknown attribute: "name"
Error uploading product Almond Croissant: Invalid documen

In [10]:
# Function to upload image and get file ID
def upload_image(file_path):
    try:
        if not os.path.exists(file_path):
            print(f"File not found: {file_path}")
            return None
            
        file_name = os.path.basename(file_path)
        print(f"Attempting to upload: {file_name}")
        
        result = storage.create_file(
            bucket_id=BUCKET_ID,
            file_id='unique()',
            file=InputFile.from_path(file_path)
        )
        print(f"Upload successful for {file_name}")
        return result['$id']
        
    except Exception as e:
        print(f"Error uploading {file_path}")
        print(f"Error type: {type(e).__name__}")
        print(f"Error details: {str(e)}")
        return None


Processing product: Cappuccino
Image path: ./products/images/cappuccino.jpg
Attempting to upload: cappuccino.jpg
Upload successful for cappuccino.jpg
Creating document for: Cappuccino
Data: {'name': 'Cappuccino', 'category': 'Coffee', 'description': 'A rich and creamy cappuccino made with freshly brewed espresso, steamed milk, and a frothy milk cap. This delightful drink offers a perfect balance of bold coffee flavor and smooth milk, making it an ideal companion for relaxing mornings or lively conversations.', 'ingredients': ['Espresso', 'Steamed Milk', 'Milk Foam'], 'price': 4.5, 'rating': 4.7, 'image_id': '6845a74977b09897eec2'}
Error processing Cappuccino
Error type: AppwriteException
Error details: Invalid document structure: Unknown attribute: "name"

Processing product: Jumbo Savory Scone
Image path: ./products/images/SavoryScone.webp
Attempting to upload: SavoryScone.webp
Upload successful for SavoryScone.webp
Creating document for: Jumbo Savory Scone
Data: {'name': 'Jumbo Savo

In [8]:
# Add this cell to create collection attributes
def setup_collection_schema():
    try:
        # Create string attributes
        database.create_string_attribute(
            database_id=DATABASE_ID,
            collection_id=COLLECTION_ID,
            key="name",
            size=255,
            required=True
        )
        
        database.create_string_attribute(
            database_id=DATABASE_ID,
            collection_id=COLLECTION_ID,
            key="category",
            size=100,
            required=True
        )
        
        database.create_string_attribute(
            database_id=DATABASE_ID,
            collection_id=COLLECTION_ID,
            key="description",
            size=1000,
            required=True
        )
        
        # Create array attribute for ingredients
        database.create_string_attribute(
            database_id=DATABASE_ID,
            collection_id=COLLECTION_ID,
            key="ingredients",
            size=100,
            array=True,
            required=True
        )
        
        # Create numeric attributes
        database.create_float_attribute(
            database_id=DATABASE_ID,
            collection_id=COLLECTION_ID,
            key="price",
            required=True
        )
        
        database.create_float_attribute(
            database_id=DATABASE_ID,
            collection_id=COLLECTION_ID,
            key="rating",
            required=True
        )
        
        # Create string attribute for image_id
        database.create_string_attribute(
            database_id=DATABASE_ID,
            collection_id=COLLECTION_ID,
            key="image_id",
            size=100,
            required=True
        )
        
        print("Collection schema setup completed successfully")
        
    except Exception as e:
        print(f"Error setting up schema: {str(e)}")

# Run the schema setup
setup_collection_schema()

Collection schema setup completed successfully


In [11]:
# Upload products with improved error handling
for index, row in df.iterrows():
    try:
        print(f"\nProcessing product: {row['name']}")
        
        # Verify all required fields are present
        required_fields = ['name', 'category', 'description', 'ingredients', 'price', 'rating']
        missing_fields = [field for field in required_fields if field not in row]
        if missing_fields:
            print(f"Skipping product {row['name']}: Missing required fields: {missing_fields}")
            continue
        
        # Upload image
        image_path = os.path.join('./products/images/', row['image_path'])
        file_id = upload_image(image_path)
        if not file_id:
            print(f"Skipping product {row['name']} due to image upload failure")
            continue
            
        # Prepare product data
        product_data = {
            'name': row['name'],
            'category': row['category'],
            'description': row['description'],
            'ingredients': row['ingredients'],
            'price': float(row['price']),
            'rating': float(row['rating']),
            'image_id': file_id
        }
        
        print(f"Creating document for: {row['name']}")
        print(f"Data: {product_data}")
        
        # Create document
        result = database.create_document(
            database_id=DATABASE_ID,
            collection_id=COLLECTION_ID,
            document_id='unique()',
            data=product_data
        )
        
        print(f"Successfully uploaded product: {row['name']}")
        
    except Exception as e:
        print(f"Error processing {row['name']}")
        print(f"Error type: {type(e).__name__}")
        print(f"Error details: {str(e)}")


Processing product: Cappuccino
Attempting to upload: cappuccino.jpg
Upload successful for cappuccino.jpg
Creating document for: Cappuccino
Data: {'name': 'Cappuccino', 'category': 'Coffee', 'description': 'A rich and creamy cappuccino made with freshly brewed espresso, steamed milk, and a frothy milk cap. This delightful drink offers a perfect balance of bold coffee flavor and smooth milk, making it an ideal companion for relaxing mornings or lively conversations.', 'ingredients': ['Espresso', 'Steamed Milk', 'Milk Foam'], 'price': 4.5, 'rating': 4.7, 'image_id': '684756d972b829d5affa'}
Successfully uploaded product: Cappuccino

Processing product: Jumbo Savory Scone
Attempting to upload: SavoryScone.webp
Upload successful for SavoryScone.webp
Creating document for: Jumbo Savory Scone
Data: {'name': 'Jumbo Savory Scone', 'category': 'Bakery', 'description': 'Deliciously flaky and buttery, this jumbo savory scone is filled with herbs and cheese, creating a mouthwatering experience. Per

Error creating bucket: Invalid `permissions` param: Invalid permission string format: "read:any".
